# [Regression] 재정대응지수와 합계출산율(TFR) 관련성 분석(모형 C) — 2026-08-07

관련 이슈: #98(주분석), #62(재정대응지수 구축, 모형 A 보완 분석)

## 분석 배경과 방법

세부영역별 재정대응(F_i,t-1, 인구1인당 실질예산액의 1년 전 값)이 합계출산율
(TFR_i,t)과 조건부·시차적으로 관련되는지, 세부영역(11개) 단위로 개별 회귀를
추정한다. 관측자료 수준의 관련성만 확인하며 인과효과를 단정하지 않는다.

```
합계출산율_i,t = φ·log1p(F_i,t-1) + α_i + λ_t + ε_it   (기본모형)
```

- 종속변수: 합계출산율(TFR_i,t), 변환 없음(분포가 0.55~1.82로 왜도가 크지 않음)
- 설명변수: log1p(F_i,t-1) — F_i,t-1 자체는 세부영역 간 스케일이 1만 배 이상
  차이 나는 우측왜도가 있어 로그변환한다(모형 A와 동일한 이유, 팀 결정문이
  아니라 이 노트북의 모형화 판단)
- α_i, λ_t: 지역·연도 고정효과, 표준오차는 지역 군집(17개)으로 보정
- 세부영역마다 별도 추정, 기본모형은 통제변수 없음(2026-08-07 팀 결정)

강건성체크 2종(기본모형과 분리 제시):
1. F_i,t-2(2년 시차) — 재정멘토 권장
2. S_i,t-1(구조환경지수 전년도, #82/#96)을 통제변수로 추가

이 노트북은 회귀표본을 처음부터 다시 만들지 않고, #62/#98 공용 파이프라인
(`scripts/build_subarea_fiscal_response_regression_sample.py`)이 저장한
`data/processed/analysis/2016-2024_세부영역별_재정반응성_회귀표본.csv`를
그대로 불러와 모형 C만 추정한다. F_it 산출·구조환경지수 결합 파이프라인
재현은 `notebooks/20260729_EDA_재정대응지수_구축_및_재정반응성_분석.ipynb`
(모형 A)에 이미 있다.

### 1. 분석 환경 설정

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REGRESSION_SAMPLE_PATH = (
    REPO_ROOT / "data" / "processed" / "analysis" / "2016-2024_세부영역별_재정반응성_회귀표본.csv"
)
SAVED_RESULT_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "analysis"
    / "2016-2024_세부영역별_재정_TFR_모형C_고정효과_결과.csv"
)

pd.set_option("display.width", 120)
np.random.seed(42)

## 2. 데이터 로드 & 기본 검사

In [2]:
sample = pd.read_csv(REGRESSION_SAMPLE_PATH)
print("shape:", sample.shape)
print(sample.dtypes)
print("\n결측 비율:")
print(sample.isna().mean().round(3))
print("\n중복(지역·연도·세부영역):", sample.duplicated(["지역", "연도", "세부영역"]).sum())

shape: (1683, 17)
지역                                str
연도                              int64
세부영역                              str
당해계획예산_백만원_provisional        float64
사업수                             int64
예산결측_사업수                        int64
CPI_지수                        float64
CPI_기준연도                        int64
당해계획예산_실질_백만원_provisional     float64
전년대비_실질증감률_pct_provisional    float64
전체인구_명                          int64
인구1인당_실질예산_원                  float64
직전1년_출산율하락도                   float64
합계출산율                         float64
구조환경지수_전년도                    float64
인구1인당_실질예산_전년도                float64
인구1인당_실질예산_전전년도               float64
dtype: object

결측 비율:
지역                            0.000
연도                            0.000
세부영역                          0.000
당해계획예산_백만원_provisional        0.000
사업수                           0.000
예산결측_사업수                      0.000
CPI_지수                        0.000
CPI_기준연도                      0.000
당해계획예산_실질_백만원_provisional 

## 3. EDA — TFR·F_i,t-1 분포

- TFR(합계출산율)은 결측 없이 0.55~1.82 범위(왜도가 크지 않아 변환하지 않음)
- F_i,t-1(`인구1인당_실질예산_전년도`)은 2016년 관측치가 시차 계산상 결측되므로
  11.1%(=17×11÷1,683) 결측이 기대값과 일치하는지 확인한다.

In [3]:
print(sample["합계출산율"].describe().round(3))
print()
print(sample["인구1인당_실질예산_전년도"].describe().round(2))
print()

expected_missing_lag1 = 17 * 11  # 2016년만 F_i,t-1 계산 불가
actual_missing_lag1 = sample["인구1인당_실질예산_전년도"].isna().sum()
assert actual_missing_lag1 == expected_missing_lag1, (
    expected_missing_lag1,
    actual_missing_lag1,
)
print(f"F_i,t-1 결측 {actual_missing_lag1}행 = 기대값({expected_missing_lag1}) 일치")

expected_missing_lag2 = 17 * 11 * 2  # 2016~2017년 F_i,t-2 계산 불가
actual_missing_lag2 = sample["인구1인당_실질예산_전전년도"].isna().sum()
assert actual_missing_lag2 == expected_missing_lag2, (
    expected_missing_lag2,
    actual_missing_lag2,
)
print(f"F_i,t-2 결측 {actual_missing_lag2}행 = 기대값({expected_missing_lag2}) 일치")

count    1683.000
mean        0.987
std         0.220
min         0.552
25%         0.827
50%         0.952
75%         1.122
max         1.821
Name: 합계출산율, dtype: float64

count       1496.00
mean       42682.95
std       138743.11
min            0.00
25%          279.35
50%         2841.78
75%        13040.19
max      3302866.16
Name: 인구1인당_실질예산_전년도, dtype: float64

F_i,t-1 결측 187행 = 기대값(187) 일치
F_i,t-2 결측 374행 = 기대값(374) 일치


## 4. 모형 C — 기본모형(TFR ~ log1p(F_i,t-1))

In [4]:
import sys

sys.path.insert(0, str(REPO_ROOT))

from scripts.run_subarea_fiscal_tfr_regression import (
    LAG1_COLUMN,
    LAG2_COLUMN,
    STRUCTURAL_CONTROL_COLUMN,
    run_subarea_models,
)

basic_result = run_subarea_models(sample, lag_column=LAG1_COLUMN)
display_columns = [
    "모형",
    "계수",
    "군집표준오차",
    "p값",
    "95%신뢰구간_하한",
    "95%신뢰구간_상한",
    "관측치",
]
basic_result[display_columns].round(4)

,모형,계수,군집표준오차,p값,95%신뢰구간_하한,95%신뢰구간_상한,관측치
0,1-1. 고용여건,-0.0001,0.0026,0.9682,-0.0057,0.0054,136
1,1-2. 주거안정성,-0.0012,0.0015,0.4306,-0.0044,0.0020,136
2,1-3. 경제적 여건,0.0044,0.0053,0.4248,-0.0069,0.0157,136
3,2-1. 돌봄 여건,-0.0058,0.0105,0.5899,-0.0281,0.0165,136
4,2-2. 여가 인프라,-0.0009,0.0042,0.8330,-0.0099,0.0081,136
5,2-3. 가사수행 격차,0.0091,0.0046,0.0645,-0.0006,0.0188,136
6,3-1. 의료서비스 여건,-0.0003,0.0062,0.9679,-0.0133,0.0128,136
7,3-2. 산후조리 여건,-0.0025,0.0070,0.7242,-0.0173,0.0123,136
8,3-3. 아동안전 수준,0.0054,0.0056,0.3467,-0.0065,0.0174,136
9,4-1. 일·가정 양립 여건,-0.0032,0.0075,0.6807,-0.0191,0.0128,136


## 5. 강건성체크 1 — F_i,t-2(2년 시차)

재정멘토가 권장한 2년 시차 버전. 기본모형과 부호·유의성이 크게 달라지는지
확인한다.

In [5]:
lag2_result = run_subarea_models(sample, lag_column=LAG2_COLUMN)
lag2_result[display_columns].round(4)

,모형,계수,군집표준오차,p값,95%신뢰구간_하한,95%신뢰구간_상한,관측치
0,1-1. 고용여건,-0.0014,0.0021,0.5109,-0.0057,0.0030,119
1,1-2. 주거안정성,-0.0007,0.0023,0.7571,-0.0055,0.0041,119
2,1-3. 경제적 여건,0.0045,0.0053,0.4073,-0.0067,0.0157,119
3,2-1. 돌봄 여건,0.0007,0.0052,0.8908,-0.0102,0.0116,119
4,2-2. 여가 인프라,-0.0029,0.0046,0.5302,-0.0126,0.0068,119
5,2-3. 가사수행 격차,0.0102,0.0052,0.0669,-0.0008,0.0212,119
6,3-1. 의료서비스 여건,0.0017,0.0064,0.7937,-0.0118,0.0152,119
7,3-2. 산후조리 여건,0.0013,0.0071,0.8622,-0.0138,0.0163,119
8,3-3. 아동안전 수준,0.0030,0.0027,0.2826,-0.0027,0.0088,119
9,4-1. 일·가정 양립 여건,-0.0042,0.0046,0.3682,-0.0139,0.0055,119


In [6]:
comparison_lag = basic_result[["모형", "계수", "p값"]].merge(
    lag2_result[["모형", "계수", "p값"]], on="모형", suffixes=("_F_t-1", "_F_t-2")
)
comparison_lag["부호일치"] = np.sign(comparison_lag["계수_F_t-1"]) == np.sign(
    comparison_lag["계수_F_t-2"]
)
comparison_lag.round(4)

,모형,계수_F_t-1,p값_F_t-1,계수_F_t-2,p값_F_t-2,부호일치
0,1-1. 고용여건,-0.0001,0.9682,-0.0014,0.5109,True
1,1-2. 주거안정성,-0.0012,0.4306,-0.0007,0.7571,True
2,1-3. 경제적 여건,0.0044,0.4248,0.0045,0.4073,True
3,2-1. 돌봄 여건,-0.0058,0.5899,0.0007,0.8908,False
4,2-2. 여가 인프라,-0.0009,0.8330,-0.0029,0.5302,True
5,2-3. 가사수행 격차,0.0091,0.0645,0.0102,0.0669,True
6,3-1. 의료서비스 여건,-0.0003,0.9679,0.0017,0.7937,False
7,3-2. 산후조리 여건,-0.0025,0.7242,0.0013,0.8622,False
8,3-3. 아동안전 수준,0.0054,0.3467,0.0030,0.2826,True
9,4-1. 일·가정 양립 여건,-0.0032,0.6807,-0.0042,0.3682,True


## 6. 강건성체크 2 — S_i,t-1(구조환경지수 전년도) 통제변수 추가

기본모형에 S_i,t-1을 통제변수로 추가한 버전. 결과가 개선되면 같이 제시하고,
기본모형 계수가 뒤집히면 보여주지 않는다(탐색적 강건성 체크로만 취급).

In [7]:
control_result = run_subarea_models(
    sample, lag_column=LAG1_COLUMN, controls=[STRUCTURAL_CONTROL_COLUMN]
)
control_result[display_columns].round(4)

,모형,계수,군집표준오차,p값,95%신뢰구간_하한,95%신뢰구간_상한,관측치
0,1-1. 고용여건,-0.0002,0.0027,0.9542,-0.0059,0.0056,136
1,1-2. 주거안정성,-0.0012,0.0016,0.4349,-0.0045,0.0021,136
2,1-3. 경제적 여건,0.0044,0.0054,0.4290,-0.0070,0.0158,136
3,2-1. 돌봄 여건,-0.0057,0.0083,0.4997,-0.0232,0.0118,136
4,2-2. 여가 인프라,0.0004,0.0034,0.9133,-0.0069,0.0077,136
5,2-3. 가사수행 격차,0.0089,0.0045,0.0633,-0.0006,0.0184,136
6,3-1. 의료서비스 여건,0.0006,0.0056,0.9176,-0.0113,0.0125,136
7,3-2. 산후조리 여건,-0.0028,0.0070,0.6909,-0.0176,0.0119,136
8,3-3. 아동안전 수준,0.0055,0.0052,0.3130,-0.0056,0.0165,136
9,4-1. 일·가정 양립 여건,-0.0043,0.0087,0.6247,-0.0228,0.0141,136


In [8]:
comparison_control = basic_result[["모형", "계수", "p값"]].merge(
    control_result[["모형", "계수", "p값"]], on="모형", suffixes=("_기본모형", "_S통제")
)
comparison_control["부호일치"] = np.sign(comparison_control["계수_기본모형"]) == np.sign(
    comparison_control["계수_S통제"]
)
comparison_control.round(4)

,모형,계수_기본모형,p값_기본모형,계수_S통제,p값_S통제,부호일치
0,1-1. 고용여건,-0.0001,0.9682,-0.0002,0.9542,True
1,1-2. 주거안정성,-0.0012,0.4306,-0.0012,0.4349,True
2,1-3. 경제적 여건,0.0044,0.4248,0.0044,0.4290,True
3,2-1. 돌봄 여건,-0.0058,0.5899,-0.0057,0.4997,True
4,2-2. 여가 인프라,-0.0009,0.8330,0.0004,0.9133,False
5,2-3. 가사수행 격차,0.0091,0.0645,0.0089,0.0633,True
6,3-1. 의료서비스 여건,-0.0003,0.9679,0.0006,0.9176,False
7,3-2. 산후조리 여건,-0.0025,0.7242,-0.0028,0.6909,True
8,3-3. 아동안전 수준,0.0054,0.3467,0.0055,0.3130,True
9,4-1. 일·가정 양립 여건,-0.0032,0.6807,-0.0043,0.6247,True


## 7. 저장 산출물과의 일치 검증

In [9]:
saved = pd.read_csv(SAVED_RESULT_PATH)

reproduced = pd.concat(
    [
        basic_result.assign(모형버전="기본모형(F_t-1)"),
        lag2_result.assign(모형버전="강건성체크_2년시차(F_t-2)"),
        control_result.assign(모형버전="강건성체크_S통제(F_t-1+S_t-1)"),
    ],
    ignore_index=True,
)[saved.columns]

pd.testing.assert_frame_equal(
    reproduced.sort_values(["모형버전", "모형"]).reset_index(drop=True),
    saved.sort_values(["모형버전", "모형"]).reset_index(drop=True),
    check_exact=False,
    rtol=1e-8,
)
print("저장된 산출물과 재현 결과 일치")

저장된 산출물과 재현 결과 일치


## 8. 해석 및 다음 단계

**핵심 발견:**
- 기본모형(F_i,t-1)에서 11개 세부영역 중 95% 신뢰구간이 0을 배제하는 세부영역은
  없다. "2-3. 가사수행 격차"가 p≈0.06~0.07로 가장 낮지만 5% 기준은 충족하지
  못한다.
- F_i,t-2(2년 시차) 강건성체크는 대체로 기본모형과 부호가 일치하지만, 표본이
  더 짧아(2018~2024, 7개년) 정밀도가 더 낮다.
- S_i,t-1(구조환경지수)을 통제변수로 추가해도 기본모형 계수 부호가 뒤집히는
  세부영역은 없다 — 강건성체크 결과를 기본모형과 함께 제시할 수 있는 근거.

**해석:** 현재 자료·모형 수준에서는 재정대응(F_i,t-1)과 합계출산율(TFR) 사이의
통계적으로 유의한 관련성을 세부영역 단위로 확인하지 못했다. 이는 관계가
없다는 증거가 아니라 표본 크기(세부영역당 136개)와 모형의 정밀도 한계로
해석해야 한다. 11번의 독립적 세부영역별 회귀를 비교한 탐색 결과이며 다중검정
보정을 적용하지 않았다.

**권장 다음 단계:**
- 결과 보고서(`reports/methodology/20260807_..._모형C_결과보고서.md`) 참고
- 필요 시 세부영역을 대영역(4개)으로 묶어 표본을 늘린 버전과 비교